# 05 — Gut Metabolic Modules (TP1 & TP2)

**Paper title:** Maternal gut microbiome diversity and functional potential in early pregnancy are associated with large-for-gestational-age birth (SweMaMi cohort)

**Purpose:** Profiles 104 Gut Metabolic Modules using two complementary approaches: MaAsLin3 (modules as outcome) and prevalence-based logistic regression (birth weight category as outcome). Modules reaching nominal p < 0.05 in both approaches are reported as differing between groups.

**Produces:** Figures 6, 7 (GMM forest plots + Venn), Supplementary Data 4

**Note:** Figure 8 (gut metabolic map) is *not* generated in R. It is a conceptual figure created manually (adapted from Vieira-Silva et al.) to illustrate the positions of the differentially associated modules identified by this analysis.


## Setup


In [ ]:
library(maaslin3)
library(dplyr)
library(data.table)
library(ggplot2)
library(tidyverse)
library(patchwork)

## 1. Load GMM abundance estimates


In [ ]:
# Define base path 
base_path <- "data"



### TP1

In [ ]:
meta_data_tp1 <- read.csv(file.path(base_path, "meta_data_tp1.csv"), 
                           header = TRUE)
head(meta_data_tp1)

In [ ]:
meta_data_tp1$Primipara <- factor(meta_data_tp1$Primipara, levels = c("0","1"))
meta_data_tp1$excess_weight <- factor(meta_data_tp1$excess_weight, levels = c("No","Yes"))
meta_data_tp1$Gest_Diabetes <- factor(meta_data_tp1$Gest_Diabetes,levels = c("0","1"))
meta_data_tp1$Q1_healthydiet <- factor(meta_data_tp1$Q1_healthydiet, levels = c("1","0"))
meta_data_tp1$Q2_healthydiet <- factor(meta_data_tp1$Q2_healthydiet, levels = c("1","0"))
meta_data_tp1$Group <- factor(meta_data_tp1$Group, levels = c("Control","Case"))

In [ ]:
row.names(meta_data_tp1) <- meta_data_tp1$kit1.faecal_sample.barcode
meta_data_tp1 <- meta_data_tp1[ , -1]




In [ ]:
module_tp1 <- read.csv(file.path(base_path, "gmm_tp1.csv"), 
                           header = TRUE,check.names = FALSE)


In [ ]:
module_tp1_no_mod <- module_tp1 %>% select(-c(Module))
row.names(module_tp1_no_mod) <- module_tp1_no_mod$Description
module_tp1_no_mod  <- module_tp1_no_mod [, -1]
head(module_tp1_no_mod)

In [ ]:
TP1_module_data <- as.data.frame(t(module_tp1_no_mod))
head(TP1_module_data)


### TP2

In [ ]:
read_meta_tp2 <- read.csv(file.path(base_path, "meta_data_tp2.csv"), 
                           header = TRUE)
meta_data_tp2 <- meta_data_tp1 %>% filter(kit2.faecal_sample.barcode %in% read_meta_tp2$kit2.faecal_sample.barcode)
head(meta_data_tp2)
dim(meta_data_tp2)

In [ ]:
row.names(meta_data_tp2) <- meta_data_tp2$kit2.faecal_sample.barcode



In [ ]:

meta_data_tp2 <- meta_data_tp2[ , -1]
head(meta_data_tp2)

In [ ]:
module_tp2 <- read.csv(file.path(base_path,"gmm_tp2.csv"), header = TRUE, check.names = FALSE)
head(module_tp2)
dim(module_tp2)

In [ ]:
module_tp2_no_mod <- module_tp2 %>% select(-c(Module))
row.names(module_tp2_no_mod) <- module_tp2_no_mod$Description
module_tp2_no_mod  <- module_tp2_no_mod [, -1]
TP2_module_data <- as.data.frame(t(module_tp2_no_mod))

## 2. MaAsLin3 across all 104 modules


In [ ]:
adjust_output_folder_TP1 <- file.path(base_path, "/Adjust_GMM_TP1")
adjust_output_folder_TP2 <- file.path(base_path, "Adjust_GMM_TP2")

In [ ]:
folder_path_1 <- "read_count_files_241217"

files_1 <- list.files(folder_path_1, pattern = "\\.txt$", full.names = TRUE)

read_depth <- files_1 %>%
  lapply(read.delim) %>%   
  bind_rows()

In [ ]:
read_depth_1<- read_depth %>%
  mutate(
    Sample_ID   = sub(".*__(\\d+)__.*", "\\1", read_depth$Sample),
    Flowcell_ID = sub(".*__(E\\d+).*", "\\1", read_depth$Sample)
  )

read_depth_2 <- read_depth_1 %>% filter(Sample_ID %in% rownames(meta_data_tp1)) %>% 
                                        select(c(Sample_ID, Flowcell_ID, after_kraken2_host_removal))
read_depth_3 <- read_depth_2 %>% rename(read_depth = after_kraken2_host_removal)
                                  

In [ ]:
read_depth_tp2_1<- read_depth %>%
  mutate(
    Sample_ID   = sub(".*__(\\d+)__.*", "\\1", read_depth$Sample),
    Flowcell_ID = sub(".*__(E\\d+).*", "\\1", read_depth$Sample)
  )

read_depth_tp2_2 <- read_depth_tp2_1 %>% filter(Sample_ID %in% rownames(meta_data_tp2)) %>% 
                                        select(c(Sample_ID, Flowcell_ID, after_kraken2_host_removal))
read_depth_tp2_3 <- read_depth_tp2_2 %>% rename(read_depth = after_kraken2_host_removal)

In [ ]:
meta_data_tp1 <- meta_data_tp1 %>%
  rownames_to_column(var = "Row.names")

In [ ]:
update_meta_data_tp1 <- full_join(
  meta_data_tp1,
  read_depth_3,
  by = c("Row.names" = "Sample_ID")
)

update_meta_data_tp1 <- update_meta_data_tp1 %>%
  column_to_rownames(var = "Row.names")


In [ ]:
meta_data_tp2 <- meta_data_tp2 %>%
  rownames_to_column(var = "Row.names")
head(meta_data_tp2)

In [ ]:
update_meta_data_tp2 <- full_join(
  meta_data_tp2,
  read_depth_tp2_3,
  by = c("Row.names" = "Sample_ID")
)

# Put it back as row names
update_meta_data_tp2 <- update_meta_data_tp2 %>%
  column_to_rownames(var = "Row.names")


### TP1

In [ ]:
head(TP1_module_data)

In [ ]:
fit_data_TP1_adjust <- maaslin3(
    input_data = TP1_module_data,
    input_metadata = update_meta_data_tp1,
    output = adjust_output_folder_TP1,

    fixed_effects = c('Group', 'age_checked', 'BMI_prior', 'Primipara', 'Q1_healthydiet','read_depth'),
    
    # Set Reference 
    reference = "Group,Control;Primipara,0;Q1_healthydiet,1",
    
    normalization = "TSS", 
    transform = "LOG", 
    correction = "BH",
    plot_summary_plot = TRUE, 
    plot_associations = TRUE)

In [ ]:
all_results_tp1 <- read.table(file.path(base_path, "Adjust_GMM_TP1/all_results.tsv"), sep = "\t", header = TRUE)
head(all_results_tp1)

In [ ]:
prevalence_sig_correct <- all_results_tp1 %>%
  filter(metadata == "Group",
         model == "prevalence",
         pval_individual < 0.05) %>%
  pull(feature)

abundance_sig_correct <- all_results_tp1 %>%
  filter(metadata == "Group",
         model == "abundance",
         pval_individual < 0.05) %>%
  pull(feature)

# True classification
both_true  <- intersect(prevalence_sig_correct,
                        abundance_sig_correct)
only_prev  <- setdiff(prevalence_sig_correct,
                      abundance_sig_correct)
only_abund <- setdiff(abundance_sig_correct,
                      prevalence_sig_correct)

cat("Both models:", length(both_true), "\n")
print(both_true)

cat("Prevalence only:", length(only_prev), "\n")
print(only_prev)

cat("Abundance only:", length(only_abund), "\n")
print(only_abund)

In [ ]:
# Direction for prevalence GMM
prevalence_direction <- all_results_tp1 %>%
  filter(metadata == "Group",
         model == "prevalence",
         feature %in% only_prev) %>%
  select(feature, coef, pval_individual) %>%
  mutate(
    direction = ifelse(coef > 0, 
                       "More prevalent in LGA", 
                       "More prevalent in AGA"),
    coef = round(coef, 3),
    pval_individual = round(pval_individual, 4)
  ) %>%
  arrange(desc(coef))

cat("PREVALENCE MODEL SPECIES\n")
print(prevalence_direction)

In [ ]:
# Direction for abundance species
abundance_direction <- all_results_tp1 %>%
  filter(metadata == "Group",
         model == "abundance",
         feature %in% only_abund) %>%
  select(feature, coef, pval_individual) %>%
  mutate(
    direction = ifelse(coef > 0,
                       "Higher abundance in LGA",
                       "Higher abundance in AGA"),
    coef = round(coef, 3),
    pval_individual = round(pval_individual, 4)
  ) %>%
  arrange(desc(coef))

cat("ABUNDANCE MODEL SPECIES\n")
print(abundance_direction)

### TP2

In [ ]:
fit_data_TP1_adjust <- maaslin3(
    input_data = TP2_module_data,
    input_metadata = update_meta_data_tp2,
    output = adjust_output_folder_TP2,
    

    fixed_effects = c('Group', 'age_checked', 'BMI_prior', 'Primipara', 'Q2_healthydiet','read_depth'),
    
    # Set Reference 
    reference = "Group,Control;Primipara,0;Q2_healthydiet,1",
    
    normalization = "TSS", 
    transform = "LOG", 
    correction = "BH",
    plot_summary_plot = TRUE, 
    plot_associations = TRUE)

In [ ]:
all_results_tp2 <- read.table(file.path(base_path,"Adjust_GMM_TP2/all_results.tsv"), sep = "\t", header = TRUE)
head(all_results_tp2)

In [ ]:
prevalence_sig_correct_2 <- all_results_tp2 %>%
  filter(metadata == "Group",
         model == "prevalence",
         pval_individual < 0.05) %>%
  pull(feature)

abundance_sig_correct_2 <- all_results_tp2 %>%
  filter(metadata == "Group",
         model == "abundance",
         pval_individual < 0.05) %>%
  pull(feature)

# True classification
both_true_2  <- intersect(prevalence_sig_correct_2,
                        abundance_sig_correct_2)
only_prev_2  <- setdiff(prevalence_sig_correct_2,
                      abundance_sig_correct_2)
only_abund_2 <- setdiff(abundance_sig_correct_2,
                      prevalence_sig_correct_2)

cat("Both models:", length(both_true_2), "\n")
print(both_true_2)

cat("Prevalence only:", length(only_prev_2), "\n")
print(only_prev_2)

cat("Abundance only:", length(only_abund_2), "\n")
print(only_abund_2)

In [ ]:
# Direction for prevalence GMM
prevalence_direction_tp2 <- all_results_tp2 %>%
  filter(metadata == "Group",
         model == "prevalence",
         feature %in% only_prev_2) %>%
  select(feature, coef, pval_individual) %>%
  mutate(
    direction = ifelse(coef > 0, 
                       "More prevalent in LGA", 
                       "More prevalent in AGA"),
    coef = round(coef, 3),
    pval_individual = round(pval_individual, 4)
  ) %>%
  arrange(desc(coef))

cat("PREVALENCE MODEL SPECIES\n")
print(prevalence_direction)

In [ ]:
# Direction for abundance species
abundance_direction_tp2 <- all_results_tp2 %>%
  filter(metadata == "Group",
         model == "abundance",
         feature %in% only_abund_2) %>%
  select(feature, coef, pval_individual) %>%
  mutate(
    direction = ifelse(coef > 0,
                       "Higher abundance in LGA",
                       "Higher abundance in AGA"),
    coef = round(coef, 3),
    pval_individual = round(pval_individual, 4)
  ) %>%
  arrange(desc(coef))

cat("ABUNDANCE MODEL SPECIES\n")
print(abundance_direction)

## 3. Logistic regression (intermediate- and high-prevalence modules)


In [ ]:
TP1_func_data <- as.data.frame(t(TP1_module_data))
TP2_func_data <- as.data.frame(t(TP2_module_data))
head(TP1_func_data )
head(TP2_func_data )

In [ ]:
#function
add_other_module <- function(df, target_sum = 1000000) {
  tot <- colSums(df)
  new_row <- pmax(target_sum - tot, 0)
  new_row2 <- as.data.frame(t(new_row))
  rownames(new_row2) <- "Other_module"
  output <- rbind(df, new_row2)
  
  return(as.data.frame(output))
}

In [ ]:
#Function 
run_module_glm <- function(module_df, module_cols, outcome = "Group", confounders) {
  
  results <- data.frame(
    Module = module_cols,
    OR = NA,
    CI_lower = NA,
    CI_upper = NA,
    p_value = NA
  )
  
  confounder_str <- paste(confounders, collapse = " + ")
  
  for (i in seq_along(module_cols)) {
    mod <- module_cols[i]
    mod_bt <- paste0("`", mod, "`")
    
    model <- glm(
      formula = as.formula(paste(outcome, "~", mod_bt, "+", confounder_str)),
      data = module_df,
      family = binomial
    )
    
    OR <- exp(coef(model)[2])
    CI <- exp(confint(model)[2, ])
    p_val <- summary(model)$coefficients[2, "Pr(>|z|)"]
    
    results[i, c("OR", "CI_lower", "CI_upper", "p_value")] <- c(OR, CI[1], CI[2], p_val)
  }
  
  results$adj_p <- p.adjust(results$p_value, method = "BH")
  
  return(results)
}

### TP1

In [ ]:
presence_percent_tp1 <- rowSums(TP1_func_data > 0) / ncol(TP1_func_data) * 100
modules_above_15_tp1 <- rownames(TP1_func_data )[presence_percent_tp1 >= 15]

In [ ]:
module_tp1_15_perecent_tp1 <- TP1_func_data[rownames(TP1_func_data) %in% modules_above_15_tp1 , ]
dim(module_tp1_15_perecent_tp1)

In [ ]:
filt_new_arrange_above_15_tp1 <- add_other_module(module_tp1_15_perecent_tp1)
head(filt_new_arrange_above_15_tp1 )

In [ ]:
modules_above_below_85 <- rownames(TP1_func_data )[presence_percent_tp1 <= 85 & presence_percent_tp1 >= 15]
modules_above_below_85
length(modules_above_below_85)

In [ ]:
modules_above_above_85_tp1 <- rownames(TP1_func_data)[presence_percent_tp1 > 85 ]
modules_above_above_85_tp1
length(modules_above_above_85_tp1)

#### Intermediate prevalence

In [ ]:
module_final_15_85_tp1 <- filt_new_arrange_above_15_tp1[rownames(filt_new_arrange_above_15_tp1) %in% modules_above_below_85,]
dim(module_final_15_85_tp1 )
head(module_final_15_85_tp1 )

In [ ]:
module_final_15_85_tp1[] <- ifelse(module_final_15_85_tp1 > 0, 1, 0)
head(module_final_15_85_tp1)

In [ ]:
module_final_15_85_logic_tp1 <- as.data.frame(t(module_final_15_85_tp1))

In [ ]:
module_final_15_85_logic_tp1$kit1.faecal_sample.barcode <- row.names(module_final_15_85_logic_tp1)
head(module_final_15_85_logic_tp1)

In [ ]:
module_final_15_85_logic_group_tp1 <- merge(module_final_15_85_logic_tp1,update_meta_data_tp1, by ="row.names" , all = TRUE)
head(module_final_15_85_logic_group_tp1)

In [ ]:
tail(module_final_15_85_logic_group_tp1)

In [ ]:
colnames(module_final_15_85_logic_group_tp1)

In [ ]:
module_subset <- module_final_15_85_logic_group_tp1 %>%
  select(1:`methanogenesis from carbon dioxide`)

In [ ]:
module_cols_tp1 <- colnames(module_subset)[-1]
module_cols_tp1

In [ ]:
# call the function run_module_glm

In [ ]:
# call the function run_module_glm
results_15_85_tp1 <- run_module_glm(
 module_df  = module_final_15_85_logic_group_tp1,
  module_cols = module_cols_tp1,
  confounders = c("age_checked", "BMI_prior", "Primipara", "Q1_healthydiet")
)

In [ ]:
View(results_15_85_tp1)

In [ ]:
write.table(
  results_15_85_tp1,
  file = file_path_taxa,
  sep = "\t",
  row.names = FALSE,
  quote = FALSE
)

In [ ]:
p_value_1 <-  results_15_85_tp1 %>% filter(p_value < 0.05)
p_value_1 

#### High prevalence

In [ ]:
module_above_85_tp1 <- filt_new_arrange_above_15_tp1[rownames(filt_new_arrange_above_15_tp1) %in% modules_above_above_85_tp1,]
dim(module_above_85_tp1)
head(module_above_85_tp1)

In [ ]:
module_above_85_t_tp1 <- as.data.frame(t(module_above_85_tp1))
head(module_above_85_t_tp1)

In [ ]:
module_final_above_85_group_tp1 <- merge(module_above_85_t_tp1,update_meta_data_tp1, by ="row.names" , all = TRUE)
head(module_final_above_85_group_tp1)

In [ ]:
module_subset_tp1 <- module_final_above_85_group_tp1 %>%
  select(1:`mucin degradation`)

In [ ]:
module_cols_11_tp1 <- colnames(module_subset_tp1)[-1]
module_cols_11_tp1 

In [ ]:
results_above_85_tp1 <- run_module_glm(
  module_df = module_final_above_85_group_tp1,
  module_cols = module_cols_11_tp1 ,
  confounders = c("age_checked", "BMI_prior", "Primipara", "Q1_healthydiet")
)

In [ ]:
head(results_above_85_tp1)

In [ ]:
p_value_11 <-  results_above_85_tp1 %>% filter(p_value < 0.05)
p_value_11 

### TP2

In [ ]:
presence_percent_tp2 <- rowSums(TP2_func_data > 0) / ncol(TP2_func_data) * 100
modules_above_15_tp2 <- rownames(TP2_func_data )[presence_percent_tp2 >= 15]
modules_above_15_tp2 

In [ ]:
module_tp2_15_perecent_tp2 <- TP2_func_data[rownames(TP2_func_data) %in% modules_above_15_tp2 , ]
dim(module_tp2_15_perecent_tp2)

In [ ]:
filt_new_arrange_above_15_tp2 <- add_other_module(module_tp2_15_perecent_tp2)
head(filt_new_arrange_above_15_tp2)

In [ ]:
modules_above_below_85_tp2 <- rownames(TP2_func_data )[presence_percent_tp2 <= 85 & presence_percent_tp2 >= 15]
modules_above_below_85_tp2
length(modules_above_below_85_tp2)

In [ ]:
modules_above_above_85_tp2 <- rownames(TP2_func_data)[presence_percent_tp2 > 85 ]
modules_above_above_85_tp2
length(modules_above_above_85_tp2)

 #### Intermediate prevalence

In [ ]:
module_final_15_85_tp2 <- filt_new_arrange_above_15_tp2[rownames(filt_new_arrange_above_15_tp2) %in% modules_above_below_85_tp2,]
dim(module_final_15_85_tp2 )
head(module_final_15_85_tp2 )

In [ ]:
module_final_15_85_tp2[] <- ifelse(module_final_15_85_tp2 > 0, 1, 0)
module_final_15_85_logic_tp2 <- as.data.frame(t(module_final_15_85_tp2))
module_final_15_85_logic_tp2$kit2.faecal_sample.barcode <- row.names(module_final_15_85_logic_tp2)
head(module_final_15_85_logic_tp2)


In [ ]:
module_final_15_85_logic_group_tp2 <- merge(module_final_15_85_logic_tp2,update_meta_data_tp2, by ="row.names" , all = TRUE)
colnames(module_final_15_85_logic_group_tp1)

In [ ]:
module_subset_tp2 <- module_final_15_85_logic_group_tp2 %>%
  select(1:`methanogenesis from carbon dioxide`)
module_cols_tp2 <- colnames(module_subset_tp2)[-1]
module_cols_tp2

In [ ]:
# call the function run_module_glm
results_15_85_tp2 <- run_module_glm(
 module_df  = module_final_15_85_logic_group_tp2,
  module_cols = module_cols_tp2,
  confounders = c("age_checked", "BMI_prior", "Primipara", "Q2_healthydiet")
)


In [ ]:
head(results_15_85_tp2)

In [ ]:
p_val_tp2_1 <- results_15_85_tp2 %>% filter(p_value	< 0.05)
p_val_tp2_1 

#### High prevalence

In [ ]:
module_above_85_tp2 <- filt_new_arrange_above_15_tp2[rownames(filt_new_arrange_above_15_tp2) %in% modules_above_above_85_tp2,]
module_above_85_t_tp2 <- as.data.frame(t(module_above_85_tp2))
module_final_above_85_group_tp2 <- merge(module_above_85_t_tp2,update_meta_data_tp2, by ="row.names" , all = TRUE)

In [ ]:
colnames(module_final_above_85_group_tp2)

In [ ]:
module_subset_2_tp2 <- module_final_above_85_group_tp2 %>%
  select(1:`mucin degradation`)

module_cols_2_tp2 <- colnames(module_subset_2_tp2)[-1]



In [ ]:
results_above_85_tp2 <- run_module_glm(
  module_df = module_final_above_85_group_tp2,
  module_cols = module_cols_2_tp2 ,
  confounders = c("age_checked", "BMI_prior", "Primipara", "Q2_healthydiet")
)

In [ ]:
p_value_2_tp2 <-  results_above_85_tp2 %>% filter(p_value < 0.05)
p_value_2_tp2 

## 4. Identify convergent modules (both approaches)


### TP1

In [ ]:
maaslin_tp1 <- c(prevalence_direction$feature, abundance_direction$feature)
lg_tp1 <- c(p_value_1$Module, p_value_11$Module)

In [ ]:
intersect(maaslin_tp1, lg_tp1)

In [ ]:
venn_data_1 <- list(
  `MaAsLin 3`        = c(prevalence_direction$feature, abundance_direction$feature),
  `Secondary Method` = c(p_value_1$Module, p_value_11$Module)
)

### TP2

In [ ]:
maaslin_tp2 <- c(prevalence_direction_tp2$feature, abundance_direction_tp2$feature)
lg_tp2 <- c(p_val_tp2_1$Module, p_value_2_tp2$Module)

In [ ]:
intersect(maaslin_tp2, lg_tp2)

In [ ]:
venn_data_2 <- list(
  `MaAsLin 3`        = c(prevalence_direction_tp2$feature, abundance_direction_tp2$feature),
  `Secondary Method` = c(p_val_tp2_1$Module, p_value_2_tp2$Module)
)

## 5. Forest plots and Venn diagrams (Figures 6, 7)


In [ ]:
masslin2_gmm_tp1 <- read.table(file.path(base_path,"Adjust_GMM_TP1/all_results.tsv"),
           sep = "\t", header = TRUE)
masslin2_gmm_tp2 <- read.table(file.path(base_path,"Adjust_GMM_TP2/all_results.tsv"),
           sep = "\t", header = TRUE)


In [ ]:
tp1_intermediate <- results_15_85_tp1
tp1_high <- results_above_85_tp1 
tp2_intermediate <- results_15_85_tp2
tp2_high <- results_above_85_tp2 

In [ ]:
#TP1
plot_data_1_tp1 <- masslin2_gmm_tp1  %>%
  filter(metadata == "Group",           
         pval_individual < 0.05) %>%    
  mutate(feature = reorder(feature, coef))

#TP2
plot_data_1_tp2 <- masslin2_gmm_tp2  %>%
  filter(metadata == "Group",           
         pval_individual < 0.05) %>%    
  mutate(feature = reorder(feature, coef))


#### Functions for the plots

In [ ]:
#Function to create a plot for maaslin3 results
make_GMM_plot1 <- function(output_df) {
  
  plot_data <- output_df %>%
    arrange(coef) %>%
    mutate(
      feature = factor(feature, levels = unique(feature)),
      Group = ifelse(coef > 0, "Case", "Control"),
      Significance = ifelse(pval_individual < 0.05,
                             "p < 0.05",
                             "p \u2265 0.05")
    )
  
  p <- ggplot(plot_data, aes(x = coef, y = feature)) +
    geom_vline(xintercept = 0, linetype = "dashed", color = "grey40", linewidth = 0.8) +
    geom_errorbarh(aes(xmin = coef - stderr, xmax = coef + stderr,
                        color = Significance),
                   height = 0.2, linewidth = 0.5) +
    geom_point(aes(fill = Group), size = 2.5, shape = 21, stroke = 0.8,
               show.legend = FALSE) +
    facet_wrap(~model, scales = "free_x") +
    geom_point(aes(fill = Group), size = 2.5, shape = 21, stroke = 0.8,
               color = "black") +   # border so the filled circle shows in legend
    scale_x_continuous(breaks = scales::pretty_breaks(n = 3)) +
    scale_color_manual(
      values = c("p < 0.05" = "black", "p \u2265 0.05" = "#E69F00"),
      name = "Significance") +
    scale_fill_manual(
      values = c("Case" = "deeppink", "Control" = "#40E0D0"),
      name = "Group association") +
    labs(x = expression(beta ~ "coefficient"), y = NULL) +
    theme_bw(base_size = 11) +
    theme(
      text             = element_text(colour = "black"),
      strip.background = element_rect(fill = "grey95"),
      strip.text       = element_text(face = "bold", colour = "black", size = 10),
      axis.title       = element_text(face = "bold", colour = "black"),
      axis.text.x      = element_text(colour = "black", size = 9),
      axis.text.y      = element_text(colour = "black", size = 11),
      legend.title     = element_text(colour = "black", size = 12, face = "bold"),
      legend.text      = element_text(colour = "black", size = 12),
      axis.title.x     = element_text(colour = "black", size = 11, face = "bold"),
      panel.grid.minor = element_blank(),
      panel.spacing    = unit(1, "lines"),
      legend.position  = "right"
    )
  
  return(p)
}

In [ ]:
# Function to create a plot for logistic regression (intermediate prevalence modules)
make_GMM_plot2 <- function(intermediate_df) {
  
  df <- intermediate_df %>%
    arrange(OR) %>%
    mutate(
      Module = factor(Module, levels = Module),
      Group = ifelse(OR > 1, "Case", "Control"),
      Significance = ifelse(p_value < 0.05,
                             "p < 0.05",
                             "p \u2265 0.05"),
      color_group = case_when(
        OR > 1 & p_value < 0.05  ~ "Case, p < 0.05",
        OR > 1 & p_value >= 0.05 ~ "Case, p \u2265 0.05",
        OR < 1 & p_value < 0.05  ~ "Control, p < 0.05",
        OR < 1 & p_value >= 0.05 ~ "Control, p \u2265 0.05"
      ),
      color_group = factor(color_group,
                           levels = c(
                             "Control, p < 0.05",
                             "Control, p \u2265 0.05",
                             "Case, p < 0.05",
                             "Case, p \u2265 0.05"
                           ))
    )
  
  p <- ggplot(df, aes(x = OR, y = Module, color = Significance)) +
    
    geom_errorbarh(
      aes(xmin = CI_lower, xmax = CI_upper),
      height = 0.2,
      linewidth = 0.5) +
    
    geom_point(
      aes(fill = Group),
      size = 2.5,
      shape = 21,
      stroke = 0.8,
      color = "black") +
    
    geom_vline(xintercept = 1,
               linetype = "dashed",
               color = "grey40",
               linewidth = 0.8) +
    
    scale_x_log10() +
    scale_fill_manual(
      values = c("Case" = "deeppink", "Control" = "#40E0D0"),
      name = "Group association",
      guide = guide_legend(order = 1)
    ) +
    scale_color_manual(
      values = c("p < 0.05" = "black", "p \u2265 0.05" = "#E69F00"),
      name = "Significance",
      guide = guide_legend(order = 2)
    ) +
    
    labs(x = "Odds Ratio (95% CI)", y = NULL) +
    
    theme_minimal() +
    theme(
      axis.text.y      = element_text(size = 11, color = "black"),
      axis.text.x      = element_text(size = 9, color = "black"),
      axis.title.x     = element_text(colour = "black", size = 11),
      panel.grid.minor = element_blank(),
      legend.title     = element_text(colour = "black", size = 12, face = "bold"),
      legend.text      = element_text(colour = "black", size = 12),
      legend.position  = "right"
    )
  
  return(p)
}

In [ ]:
# Function to create a plot for logistic regression (high prevalence modules)
make_GMM_plot3 <- function(tp_high_df) {
  
  plot_df <- tp_high_df %>%
    arrange(OR) %>%
    mutate(
      Module = factor(Module, levels = Module),
      Group = ifelse(OR > 1, "Case", "Control"),
      Significance = ifelse(p_value < 0.05,
                             "p < 0.05",
                             "p \u2265 0.05")
    )
  
  plot_df_sig <- plot_df %>%
    filter(Significance == "p < 0.05")
  
  p <- ggplot(plot_df_sig,
              aes(x = OR, y = Module, color = Significance)) +
    
    geom_errorbarh(
      aes(xmin = CI_lower, xmax = CI_upper),
      height = 0.2,
      linewidth = 0.5) +
    
    geom_point(
      aes(fill = Group),
      size = 2.5,
      shape = 21,
      stroke = 0.8,
      color = "black") +
    
    geom_vline(xintercept = 1,
               linetype = "dashed",
               color = "grey40",
               linewidth = 0.8) +
    
    scale_x_log10() +
    
    scale_color_manual(
      values = c("p < 0.05" = "black",
                 "p \u2265 0.05" = "#E69F00"),
      name = "Significance"
    ) +
    
    scale_fill_manual(
      values = c("Case" = "deeppink",
                 "Control" = "#40E0D0"),
      name = "Group association"
    ) +
    
    labs(x = "Odds Ratio (95% CI)", y = NULL) +
    
    theme_minimal() +
    theme(
      axis.text.y      = element_text(size = 11, color = "black"),
      axis.text.x      = element_text(size = 9, color = "black"),
      axis.title.x     = element_text(colour = "black", size = 11),
      panel.grid.minor = element_blank(),
      legend.title     = element_text(colour = "black", size = 12, face = "bold"),
      legend.text      = element_text(colour = "black", size = 12),
      legend.position  = "right"
    )
  
  return(p)
}

#### Call functions

In [ ]:
GMM_tp1_plot_1 <- make_GMM_plot1(plot_data_1_tp1)
GMM_tp1_plot_1
GMM_tp2_plot_1 <- make_GMM_plot1(plot_data_1_tp2)
GMM_tp2_plot_1

In [ ]:
GMM_tp1_plot_2 <- make_GMM_plot2(tp1_intermediate)
GMM_tp1_plot_2
GMM_tp2_plot_2 <- make_GMM_plot2(tp2_intermediate)
GMM_tp2_plot_2

In [ ]:
GMM_tp1_plot_3 <- make_GMM_plot3(tp1_high)
GMM_tp1_plot_3

GMM_tp2_plot_3 <- make_GMM_plot3(tp2_high)
GMM_tp2_plot_3

In [ ]:
venn_plot_1 <- ggvenn(
  venn_data_1,
  fill_color    = c("white", "white"),   
  fill_alpha    = 0,                      
  stroke_color  = "black",
  stroke_size   = 0.6,
  set_name_size = 5.0,                       
  text_size     = 4.0,                       
  show_percentage = TRUE
)
venn_plot_1

In [ ]:
venn_plot_2 <- ggvenn(
  venn_data_2,
  fill_color    = c("white", "white"),   
  fill_alpha    = 0,                      
  stroke_color  = "black",
  stroke_size   = 0.6,
  set_name_size = 5.0,                       
  text_size     = 4.0,                       
  show_percentage = TRUE
)
venn_plot_2

#### Final figure (Figure 6,7)

In [ ]:
A1 <- GMM_tp1_plot_3 + ggtitle(NULL)  
B1 <- GMM_tp1_plot_2 + ggtitle(NULL)
C1 <- GMM_tp1_plot_3+ ggtitle(NULL)
D1 <- venn_plot_1                  

In [ ]:
tp1_fig_6 <- (A1 | B1) / (C1 | D1) +
  plot_layout(widths = c(1, 1)) +
  plot_annotation(
    tag_levels = list(c(
      "A) MaAsLin3: significant GMM associations (p < 0.05) at TP1",
      "B) Logistic regression: intermediate prevalence modules (15–85%) at TP1",
      "C) Logistic regression: high prevalence modules (>85%) (p < 0.05) at TP1",
      "D) Convergent GMM findings (p < 0.05) at TP1"
    ))
  ) &
  theme(plot.tag = element_text(size = 14, face = "bold", hjust = 0),
        plot.tag.position = c(0, 1),
        plot.margin = margin(t = 35, r = 10, b = 10, l = 10),   # ← more top space, equal on all
        text = element_text(size = 12))

tp1_fig_6

In [ ]:
A2 <- GMM_tp2_plot_3 + ggtitle(NULL)  
B2 <- GMM_tp2_plot_2 + ggtitle(NULL)
C2 <- GMM_tp2_plot_3+ ggtitle(NULL)
D2 <- venn_plot_2                 

In [ ]:
tp2_fig_7 <- (A2 | B2) / (C2 | D2) +
  plot_layout(widths = c(1, 1)) +
  plot_annotation(
    tag_levels = list(c(
      "A) MaAsLin3: significant GMM associations (p < 0.05) at TP2",
      "B) Logistic regression: intermediate prevalence modules (15–85%) at TP2",
      "C) Logistic regression: high prevalence modules (>85%) (p < 0.05) at TP2",
      "D) Convergent GMM findings (p < 0.05) at TP2"
    ))
  ) &
  theme(plot.tag = element_text(size = 14, face = "bold", hjust = 0),
        plot.tag.position = c(0, 1),
        plot.margin = margin(t = 35, r = 10, b = 10, l = 10),  
        text = element_text(size = 12))

tp2_fig_7